# Market Data Preprocessing

This notebook processes Ethereum historical market data from CoinMarketCap into daily aggregated features.

**Features extracted:**
- **Price**: Open, High, Low, Close (OHLC)
- **Volume & Market Cap**: Daily trading volume and market capitalization
- **Derived Features**:
  - Daily Return (% change)
  - Intraday Volatility ((High - Low) / Open)
  - 7-Day Rolling Volatility

In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_csv('../data/raw/market/ethereum_historical_data_coinmarketcap.csv', sep=';')

print(f"shape: {df.shape}")
print(f"columns: {df.columns.tolist()}")
df.head()

shape: (3685, 13)
columns: ['timeOpen', 'timeClose', 'timeHigh', 'timeLow', 'name', 'open', 'high', 'low', 'close', 'volume', 'marketCap', 'circulatingSupply', 'timestamp']


,timeOpen,timeClose,timeHigh,timeLow,name,open,high,low,close,volume,marketCap,circulatingSupply,timestamp
0,2025-09-07T00:00:00.000Z,2025-09-07T23:59:59.999Z,2025-09-07T23:22:00.000Z,2025-09-07T18:44:00.000Z,2781,4274.166805,4334.274408,4271.534091,4305.347668,1.742678e+10,5.196815e+11,1.207051e+08,2025-09-07T23:59:59.999Z
1,2025-09-06T00:00:00.000Z,2025-09-06T23:59:59.999Z,2025-09-06T00:48:00.000Z,2025-09-06T16:22:00.000Z,2781,4306.973228,4327.439778,4244.755078,4274.242063,1.810825e+10,5.159139e+11,1.207051e+08,2025-09-06T23:59:59.999Z
2,2025-09-05T00:00:00.000Z,2025-09-05T23:59:59.999Z,2025-09-05T12:32:00.000Z,2025-09-05T15:04:00.000Z,2781,4298.837052,4484.361456,4258.049765,4306.988932,4.416374e+10,5.198742e+11,1.207052e+08,2025-09-05T23:59:59.999Z
3,2025-09-04T00:00:00.000Z,2025-09-04T23:59:59.999Z,2025-09-04T00:36:00.000Z,2025-09-04T19:48:00.000Z,2781,4450.215913,4483.451012,4268.588771,4298.744222,3.491980e+10,5.188925e+11,1.207053e+08,2025-09-04T23:59:59.999Z
4,2025-09-03T00:00:00.000Z,2025-09-03T23:59:59.999Z,2025-09-03T17:34:00.000Z,2025-09-03T01:40:00.000Z,2781,4324.696399,4489.198505,4286.206097,4450.388959,3.526087e+10,5.371651e+11,1.207054e+08,2025-09-03T23:59:59.999Z


### Preprocessing and Cleaning

In [3]:
# convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['day'] = pd.to_datetime(df['timestamp'].dt.date)

# sort by date ascending
df = df.sort_values('day').reset_index(drop=True)

# select and rename relevant columns
market_df = df[['day', 'open', 'high', 'low', 'close', 'volume', 'marketCap']].copy()

# verify date range
print(f"date Range: {market_df['day'].min()} to {market_df['day'].max()}")

# check for duplicates
print(f"duplicates: {market_df['day'].duplicated().sum()}")
market_df.head()

date Range: 2015-08-07 00:00:00 to 2025-09-07 00:00:00
duplicates: 0


,day,open,high,low,close,volume,marketCap
0,2015-08-07,2.831620,3.536610,2.521120,2.772120,164329.0,1.666106e+08
1,2015-08-08,2.793760,2.798810,0.714725,0.753325,674188.0,4.548689e+07
2,2015-08-09,0.706136,0.879810,0.629191,0.701897,532170.0,4.239957e+07
3,2015-08-10,0.713989,0.729854,0.636546,0.708448,405283.0,4.281836e+07
4,2015-08-11,0.708087,1.131410,0.663235,1.067860,1463100.0,6.456929e+07


### Feature extraction and Engineering

In [4]:
# Daily Price Returns - (today_close - yesterday_close) / yesterday_close
market_df['market_daily_return'] = market_df['close'].pct_change().fillna(0)
# Standard deviation of daily price returns over the past 7 days
market_df['market_volatility_7d'] = market_df['market_daily_return'].rolling(window=7).std().fillna(0)
market_df['market_volume_spike'] = market_df['volume'] / market_df['volume'].rolling(7, min_periods=3).mean().replace(0, 1)
market_df['market_momentum_7d'] = (market_df['close'] - market_df['close'].shift(7)) / market_df['close'].shift(7)
# Intraday Volatility - How much the price moved within a single day
market_df['market_intraday_volatility'] = (market_df['high'] - market_df['low']) / market_df['open']

print("derived features created.")
market_df.tail()

derived features created.


,day,open,high,low,close,volume,marketCap,market_daily_return,market_volatility_7d,market_volume_spike,market_momentum_7d,market_intraday_volatility
3680,2025-09-03,4324.696399,4489.198505,4286.206097,4450.388959,3.526087e+10,5.371651e+11,0.028905,0.019191,0.994514,-0.011770,0.046938
3681,2025-09-04,4450.215913,4483.451012,4268.588771,4298.744222,3.491980e+10,5.188925e+11,-0.034074,0.022690,0.989381,-0.046245,0.048281
3682,2025-09-05,4298.837052,4484.361456,4258.049765,4306.988932,4.416374e+10,5.198742e+11,0.001918,0.019616,1.265302,-0.012193,0.052645
3683,2025-09-06,4306.973228,4327.439778,4244.755078,4274.242063,1.810825e+10,5.159139e+11,-0.007603,0.019601,0.535858,-0.022841,0.019198
3684,2025-09-07,4274.166805,4334.274408,4271.534091,4305.347668,1.742678e+10,5.196815e+11,0.007277,0.019857,0.536693,-0.019287,0.014679


In [5]:
selected_features = [
    'day',
    'market_daily_return',
    'market_volatility_7d',
    'market_volume_spike',
    'market_momentum_7d',
    'market_intraday_volatility'
]

market_df = market_df[selected_features].copy()
market_df = market_df.fillna(0)

In [6]:
output_dir = '../data/processed/market'
os.makedirs(output_dir, exist_ok=True)

parquet_path = os.path.join(output_dir, 'market_daily_cleaned.parquet')
market_df.to_parquet(parquet_path, index=False)
print("Saved")

Saved
